# 00 · Modelling helper: the foundations' artifacts, for reading the code

**What it is for.** A reference to keep open while you read Stage 3's foundations
(`src/core_modelling/model/`, `dataset/folds.py`, `runs.py`) and Training (`training/`)
beside `docs/components/model_bundle.md`. It calls each function on the use case's own train
rows and shows what comes back: each artifact's grain, its columns and a few telling rows,
beside the function that wrote it.

**It is not a stage.** It trains no model of the use case and records nothing: the models are
small fixed configurations on a handful of features, the bundle and the runs go to a temporary
folder that the last cell removes, and no audit-log entry is written. Nothing is written into the
use case's data folders.

**Before you start.** It needs the saved model input (`01_target_factory`) and, for the same data
definition, the feature ledger and the decision audit log (`02_eda` and `03_feature_checks`): the
ledger says which features are in force and of which kind, the audit log which are held or
excluded, and the two stage jsons give the run tokens a run records. It starts a Spark session
for the training frame and for `score_spark`. Sections 7 to 9 read a trained run:
run `04_training` first; section 10 reads its evaluation: run `05_evaluation` too.

**How it is laid out**, in the order Training calls the functions:
1. the features used here;
2. the training frame and the client folds;
3. the encoder;
4. the four model families;
5. the bundle and scoring;
6. the run record;
7. a trained run's `training.json`;
8. Training's tables;
9. the per-run ledger and `stage2/`;
10. an evaluated run: `evaluation.json`, its tables and the scores.

Each section says what the artifact
is, which function writes it (module and function under `src/core_modelling/`, so you can jump to
the code), its grain and columns, and what right looks like.

**Docs:** `docs/components/model_bundle.md` (the bundle, the manifest, `run.json`, the encoder,
the calibration, the hashes) · `docs/components/dataset.md` → "Client folds" ·
`docs/components/training.md` ("What Training writes") ·
`docs/components/evaluation.md` ("What Evaluation writes").

## Setup
The same names as `03_feature_checks`. `resolve_config` replays the decision audit log onto the
config, as every stage does first: its `features.held` and `features.excluded` are what Training
leaves out. `TEMP` is the temporary folder for everything this notebook writes.

`FAMILY` is the family whose bundle section 5 writes: change it to see another's.

In [ ]:
import hashlib
import json
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import core_modelling
from core_modelling.artifacts import read_stage, stage_json
from core_modelling.config import load_config
from core_modelling.config.audit_log import resolve_config
from core_modelling.config.identities import feature_signature, recipe_hash
from core_modelling.dataset import build_frames
from core_modelling.environment import feature_dir, get_spark_session, load_environment, models_dir
from core_modelling.ledger import read_ledger, resolve_features
from core_modelling.model import (
    FAMILIES,
    apply_calibration,
    check_bundle,
    check_monotone,
    encode_frame,
    fit_encoder,
    load_bundle,
    score_frame,
    score_spark,
    write_bundle,
)
from core_modelling.model.families import check_params
from core_modelling.model.family_inputs import check_order
from core_modelling.runs import create_run, find_run, list_runs, read_run, show_run
from core_modelling.training.ledger_run import read_run_ledger
from core_modelling.training.search import split_budget
from core_modelling.training.tables import sample_contributions

# The use case and where the framework runs, as absolute paths (as in 03_feature_checks).
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"  # or conf/base/td_acquisition.yml
ENTITY = "entity_a"
ENVIRONMENT = str(REPO / "conf/local/environment.yml")  # or None: $CORE_MODELLING_ENVIRONMENT
FAMILY = "lgbm"  # the bundle's family in section 5: any key of FAMILIES

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

env = load_environment(ENVIRONMENT)
spark = get_spark_session(env)
resolved = resolve_config(load_config(CONFIG), ENTITY, env)
training = resolved.training
TEMP_DIR = tempfile.TemporaryDirectory(prefix="modelling_helper_")
TEMP = Path(TEMP_DIR.name)
print(resolved.use_case, resolved.task, ENTITY, training.objective, training.seed, training.folds)

## 1. The features used here
Training's feature set is `resolve_features(ledger, resolved)` (`ledger.py`): the candidates minus
the lineage, unsupported, excluded and held features, in the ledger's order. This notebook takes a
handful of them so that every fit is quick: the first four numeric and the first two categorical
features in force, by the ledger's `kind`. No cell names a feature, so they work on any use case.

A held or excluded feature is refused by Training's frame: `build_frames(..., "training")` fails
naming it, since for Training a held feature acts as an excluded one until a person keeps or
excludes it.

**Look at:** the six features with their kinds, and how many features are in force, held and
excluded.

In [ ]:
folder = feature_dir(env, resolved, ENTITY)
ledger = read_ledger(folder, resolved.housekeeping["data_definition_hash"])
in_force = resolve_features(ledger, resolved)
kinds = ledger.set_index("feature")["kind"].to_dict()
numeric = [name for name in in_force if kinds[name] == "numeric"][:4]
categorical = [name for name in in_force if kinds[name] == "categorical"][:2]
chosen = numeric + categorical
display(ledger.set_index("feature").loc[chosen, ["kind", "source", "levels", "concordance"]])
counts = {
    "ledger rows": len(ledger),
    "in force": len(in_force),
    "held": len(resolved.features.held),
    "excluded": len(resolved.features.excluded),
}
display(pd.Series(counts).to_frame("features"))

## 2. The training frame and the client folds
`build_frames(resolved, ENTITY, env, spark, "training", features=chosen)` (`dataset/frames.py`)
returns `Frames`, whose `train` is Training's frame, collected: **one row per client and period of
the observation window's train split**, with `unit_id`, `period`, the features asked for,
`target`, `split` and `cv_fold` (and `cv_fold_2`, … when `training.fold_repeats` is above 1).
Test and validation rows cannot reach it: the rows are cut to the window and to
`split = 'train'` in Spark before anything else, and the validation period lies after the window.

`cv_fold` is dealt in Spark by `assign_folds` (`dataset/folds.py`), after the sampling (none in
this config) and before the collect:
- a fold holds whole clients: every row of a client has the client's fold;
- for a binary task the converting clients (a positive row in train) are put in the order of a
  salted hash of their id and dealt to the folds in turn; every other client takes that hash cut
  into K folds (`training.folds`);
- the salt is the dealing's own with `training.seed`, so the folds are independent of the
  train/test split and of the sampling, and a rerun repeats them.

`meta["folds"]` is `fold_counts`' table, checked by `check_folds` before the collect: **one row
per dealing and fold**, with `rows`, `clients`, `converting_clients` and `positive_rows`. A fold
without a train row, or for a binary task without a positive row, stops Training.

**Right looks like:** every client in one fold; the folds' converting-client counts differ by at
most one; each dealing's rows add up to the frame's; `split` holds `train` only; the periods are
the window's.

In [ ]:
frames = build_frames(resolved, ENTITY, env, spark, "training", features=chosen)
train = frames.train
folds = frames.meta["folds"]
display(train.dtypes.to_frame("dtype").T, train.head(6))
display(folds)
by_dealing = folds.groupby("dealing")
spread = by_dealing["converting_clients"].max() - by_dealing["converting_clients"].min()
checks = {
    "folds per client, at most": train.groupby("unit_id")["cv_fold"].nunique().max(),
    "converting clients, largest fold minus smallest": spread.max(),
    "each dealing's rows equal the frame's": (by_dealing["rows"].sum() == len(train)).all(),
    "split values": sorted(train["split"].unique()),
    "periods": f"{train['period'].min()} to {train['period'].max()}",
}
display(pd.Series(checks).to_frame("check"))

## 3. The encoder
`fit_encoder(train, features, kinds, order, min_level_client_count, unordered)`
(`model/encoder.py`) reads the train frame only and returns plain data: the encoder's settings
and **one entry per feature, in the model's column order**, with `name`, `kind` and, for a
categorical, `levels` (and `pooled` for an unordered one). `encode_frame(frame, encoder)` applies
it to any frame and returns the encoded frame (the same rows and index, exactly the encoder's
features in its order) and **one row of shares per feature**: `null_share`, `infinite_share`
(numeric), `other_share` (unordered) and `unseen_share` (categorical), null where a share does not
apply to the kind.

The code rule per kind:
- **numeric**: the value as float64, clipped to ±1e30 so an infinity stays apart from null; a null
  stays null.
- **ordered** (a categorical named in `features.order`): the level's position in the declared list
  (0, 1, …) as float64; a null stays null, and a level missing from the list becomes null and
  counts as unseen. A train level missing from the list fails the fit: the list is a contract.
- **unordered** (every other categorical): 0 for a null; 1 … K for the levels that at least
  `min_level_client_count` distinct train clients hold, most rows first; K + 1 for `__other__`,
  which takes the rarer train levels (`pooled`) and any level unseen in train.

Nothing is imputed, and the levels are the only fitted statistic. Levels are compared as text by
EDA's rule (an integral number without `.0`, a boolean in lower case), so a declared order is
written in that form.

**The order here is a demonstration.** This config declares no order, so the cell fits once
without one, takes the last chosen categorical's levels as found, sorts them as text, and fits
again with that list as its order, so that all three kinds appear. The sorted order is not a
claim about the data.

**Look at:** one entry of each kind; then a few rows raw beside encoded, rows with a null first (a
null numeric stays null, a null unordered level is 0); the unordered feature's rows per code,
from 0 (null) to K + 1 (`__other__`), a code without rows shown as 0; the shares, whose
`other_share` and `unseen_share` match those rows.

In [ ]:
rule = training.categoricals
first = fit_encoder(train, chosen, kinds, {}, rule.min_level_client_count, rule.unordered)
ordered = categorical[-1]
found = next(entry for entry in first["features"] if entry["name"] == ordered)
order = {ordered: sorted(found["levels"] + found["pooled"])}  # a demonstration only
encoder = fit_encoder(train, chosen, kinds, order, rule.min_level_client_count, rule.unordered)
display({key: value for key, value in encoder.items() if key != "features"})
display(pd.DataFrame(encoder["features"]).drop_duplicates("kind").set_index("name"))

In [ ]:
encoded, shares = encode_frame(train, encoder)
with_null = train.index[train[chosen].isna().any(axis=1)][:3]
complete = train.index[train[chosen].notna().all(axis=1)][:3]
picked = [*with_null, *complete]
display(pd.concat({"raw": train.loc[picked, chosen], "encoded": encoded.loc[picked]}, axis=1))
unordered = next(entry for entry in encoder["features"] if entry["kind"] == "unordered")
every_code = range(len(unordered["levels"]) + 2)  # 0 (null), 1 … K, K + 1 (__other__)
codes = encoded[unordered["name"]].value_counts().reindex(every_code, fill_value=0)
display(codes.to_frame(f"rows of {unordered['name']} by code (K = {len(unordered['levels'])})"))
display(shares)

## 4. The four model families
`FAMILIES` (`model/families.py`) maps FLAML's learner names to one set of plain functions per
library (`model/lgbm_family.py`, `xgb_family.py`, `catboost_family.py`): `fixed_params`, `fit`,
`predict`, `contributions`, `describe`, `save` and `load`. `xgboost` and `xgb_limitdepth` grow one
kind of booster: leaf by leaf under a leaf limit, or level by level under a depth limit.

Per family the cell does what Training does with one searched configuration:
1. `fixed_params(task, objective, encoder, monotone, seed, threads, tweedie_variance_power)`: the
   constants every fit of a run shares (the objective, the seed, the threads, the declared
   categoricals, the monotone constraints in the library's own form), merged with a small fixed
   configuration (`SEARCHED`: 40 trees, a learning rate of 0.1 and a size limit);
   `check_params` refuses a parameter name the library would ignore in silence;
2. `check_order` (`model/family_inputs.py`), once before any fit: the encoded frame's columns are
   the encoder's features, in its order;
3. `fit` on the encoded train frame; `save` writes the library's own file (LightGBM text, XGBoost
   JSON, CatBoost `.cbm`), never a pickle; `load` builds the model back from the file's bytes.

**The constraint is a demonstration:** the numeric with the most distinct values is declared
`increasing`, so that `check_monotone` has something to check. It says nothing about the data.

**The table, one row per family:**
- `trees`: the tree count `describe` reads from the model itself;
- `margin_min`, `margin_max`: the train rows' margins (the log-odds for a binary task);
- `reload_identical`: the model built from the saved bytes gives the same margins, bit for bit;
- `file_sha256`: the first 12 characters of the saved file's SHA-256; a repeated fit writes the
  same file at any thread count;
- `contributions_sum`: the library's own per-feature contributions plus the base value add up to
  the margin, on the first 2,000 rows (they feed the explanation views);
- `monotone`: `check_monotone` swept the constrained feature over its quantiles on 200 rows and
  found no step against the direction (it fails loud otherwise).

**Right looks like:** 40 trees per family and every check true.

In [ ]:
MONOTONE = {train[numeric].nunique().idxmax(): "increasing"}  # a demonstration only
SEARCHED = {  # a small fixed configuration per family, in FLAML's parameter names
    "lgbm": {"n_estimators": 40, "learning_rate": 0.1, "num_leaves": 15},
    "xgboost": {"n_estimators": 40, "learning_rate": 0.1, "max_leaves": 15},
    "xgb_limitdepth": {"n_estimators": 40, "learning_rate": 0.1, "max_depth": 4},
    "catboost": {"n_estimators": 40, "learning_rate": 0.1, "depth": 4},
}
X, y = encoded, train["target"].to_numpy()
check_order(X, encoder)
head = X.head(2000)
params, boosters, rows = {}, {}, []
for name, family in FAMILIES.items():
    fixed = family.fixed_params(
        resolved.task,
        training.objective,
        encoder,
        MONOTONE,
        training.seed,
        1,
        training.tweedie_variance_power,
    )
    params[name] = {**fixed, **SEARCHED[name]}
    check_params(name, params[name])
    boosters[name] = booster = family.fit(X, y, params[name], resolved.task)
    path = TEMP / f"{name}_{family.file_name}"
    family.save(booster, str(path))
    margin = family.predict(booster, X)
    reloaded = family.load(path.read_bytes())
    check_monotone(name, booster, X, MONOTONE)  # fails loud on a step against the direction
    contributions = family.contributions(booster, head).sum(axis=1)
    rows.append(
        {
            "family": name,
            "file": family.file_name,
            "trees": family.describe(booster)["trees"],
            "margin_min": margin.min(),
            "margin_max": margin.max(),
            "reload_identical": np.array_equal(margin, family.predict(reloaded, X)),
            "file_sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12],
            "contributions_sum": np.allclose(
                contributions, family.predict(booster, head), rtol=1e-5, atol=1e-5
            ),
            "monotone": True,
        }
    )
display(pd.DataFrame(rows).set_index("family"))

## 5. The bundle and scoring
`write_bundle(folder, family, booster, content, identity)` (`model/bundle.py`) checks what it is
given, writes three files into a new folder and returns the `model_hash`:
- **the model file**, through the family's `save`;
- **`bundle.json`**: the caller's `content` (`task`, `features`, `encoder`, `calibration`,
  `recipe`) plus `bundle_schema_version`, `family`, `model_file`, `output_columns` and
  `library_version`. It holds no time, run or person and has one fixed layout, so the same model
  always gives the same bytes;
- **`manifest.json`**: `model_hash` (one SHA-256 over the model file and `bundle.json`, each
  file's name and length with its bytes), `recipe_hash` and `feature_signature` with their
  `hash_version`, the run's identity, `framework_version` and `written_at`.

It refuses a content with other keys, features that are not the encoder's, a calibration it
cannot apply (one that would reverse the model's order among them), and a recipe with other keys
than `task`, `family`, `parameters` (the seed and the tree count among them, never a thread
count), `features`, `encoder`, `monotone`, `calibration` (its method and folds, never its fitted
numbers), `split` (the client split), `sampling` and `data_definition_hash`, or whose task,
family, features or encoder differ from the content's.

**The calibration here is the identity:** Platt with `a = 1`, `b = 0` and a zero offset, so
`score = sigmoid(1 · margin + 0 + 0)` equals `score_raw = sigmoid(margin)`. It is chosen so that
the three columns read plainly: `margin` the model's log-odds, `score_raw` its probability,
`score` the calibrated probability. Training fits the real numbers on out-of-fold margins, and
the offset undoes a declared sampling. Both example configs are binary; a regression bundle's
calibration has the link `identity` or `log` and the method `mean`.

**The identity belongs to no run:** `run_id` is the placeholder `r000`, a number no run is ever
given; the three hashes are the resolved config's.

**Look at:** the three files; `bundle.json`'s keys with their values shortened; the whole
manifest; the hashes recomputed from the recipe and the feature list.

In [ ]:
CALIBRATION = {"link": "logit", "method": "platt", "parameters": {"a": 1.0, "b": 0.0},
               "offset": 0.0}  # fmt: skip
features = [entry["name"] for entry in encoder["features"]]
recipe = {
    "task": resolved.task,
    "family": FAMILY,
    # the thread count is the machine's, not the model's: it stays out of the recipe
    "parameters": {key: value for key, value in params[FAMILY].items() if key != "n_jobs"},
    "features": features,
    "encoder": encoder,
    "monotone": MONOTONE,
    "calibration": {"method": CALIBRATION["method"], "folds": training.folds},
    "split": resolved.data.split.model_dump(mode="json"),
    "sampling": None if training.sampling is None else training.sampling.model_dump(mode="json"),
    "data_definition_hash": resolved.housekeeping["data_definition_hash"],
}
content = {"task": resolved.task, "features": features, "encoder": encoder,
           "calibration": CALIBRATION, "recipe": recipe}  # fmt: skip
hashes = ("data_definition_hash", "config_hash", "resolved_input_hash")
identity = {"run_id": "r000", **{key: resolved.housekeeping[key] for key in hashes}}
bundle_folder = str(TEMP / "bundle")
model_hash = write_bundle(bundle_folder, FAMILY, boosters[FAMILY], content, identity)
bundle_json = json.loads(Path(bundle_folder, "bundle.json").read_text())
manifest = json.loads(Path(bundle_folder, "manifest.json").read_text())
display(sorted(path.name for path in Path(bundle_folder).iterdir()))
display(
    pd.DataFrame({"value": {key: json.dumps(value)[:100] for key, value in bundle_json.items()}})
)
display(manifest)
recomputed = {
    "write_bundle returned the manifest's model_hash": model_hash == manifest["model_hash"],
    "recipe_hash(recipe)": recipe_hash(recipe) == manifest["recipe_hash"],
    "feature_signature(features)": feature_signature(features) == manifest["feature_signature"],
}
display(pd.Series(recomputed).to_frame("equals the manifest's"))

`load_bundle(folder)` recomputes `model_hash` from the files before it parses anything else,
refuses a difference, and builds the model from the very bytes it verified: a `Bundle` with the
model, the encoder, the calibration and the features. `score_frame(bundle, frame)`
(`model/scoring.py`) encodes raw feature values with the bundle's encoder, predicts the margin
through the family and applies the calibration: **one row per input row, on its index**, with
`margin`, `score_raw` and `score`, all float64. A missing feature or a non-finite score fails
loud.

`check_bundle(folder, sample, expected)` is Training's self-check at save: the bundle reloaded
from disk must score a fixed sample of train rows exactly as the model in memory does (here
`expected` is encode, `predict` and `apply_calibration` on the booster fitted in section 4).

**Right looks like:** `score` equal to `score_raw` (the identity calibration), a row with a null
feature scored like any other, and `check_bundle` returning without a word.

In [ ]:
bundle = load_bundle(bundle_folder)
raw = train.loc[picked, ["unit_id", "period", *chosen]]
display(pd.concat([raw, score_frame(bundle, raw)], axis=1))
sample = train.head(1000)
margin = FAMILIES[FAMILY].predict(boosters[FAMILY], encode_frame(sample, encoder)[0])
score_raw, score = apply_calibration(margin, CALIBRATION, resolved.task)
expected = pd.DataFrame({"margin": margin, "score_raw": score_raw, "score": score},
                        index=sample.index)  # fmt: skip
check_bundle(bundle_folder, sample, expected)  # silent: the bundle scores as the model in memory
loaded = {
    "family": bundle.family,
    "model_hash": bundle.model_hash[:12],
    "features": bundle.features,
}
display(pd.Series(loaded).to_frame("load_bundle"))

`score_spark(folder, rows, keep)` scores a Spark frame with the same `score_frame`, per
partition through `mapInPandas`: the driver reads and verifies the bundle's bytes and ships them,
never a path, and each partition builds the model once. It returns **one row per input row**: the
`keep` columns and the three scores. The cell scores a few hundred train rows both ways.

**Right looks like:** the same clients in the same order, and identical scores.

In [ ]:
keys = ["unit_id", "period"]
few = train.head(300)[[*keys, *chosen]]
in_spark = score_spark(bundle_folder, spark.createDataFrame(few), keys).toPandas()
in_pandas = pd.concat([few[keys], score_frame(bundle, few)], axis=1)
in_spark = in_spark.sort_values(keys, ignore_index=True)
in_pandas = in_pandas.sort_values(keys, ignore_index=True)
scores = ["margin", "score_raw", "score"]
compared = {
    "rows": len(in_spark),
    "same clients in the same order": (
        in_spark["unit_id"].to_numpy() == in_pandas["unit_id"].to_numpy()
    ).all(),
    "identical scores": np.array_equal(in_spark[scores].to_numpy(), in_pandas[scores].to_numpy()),
}
display(pd.Series(compared).to_frame("score_spark against score_frame"))

**A changed byte is refused.** The cell copies the bundle, flips one bit in the middle of the
copy's model file and loads the copy: `load_bundle` refuses it before it builds the model, naming
the folder and both hashes. The `try` only displays the refusal's message; had the copy loaded,
the cell would fail.

In [ ]:
tampered = TEMP / "tampered"
shutil.copytree(bundle_folder, tampered)
model_file = tampered / bundle.content["model_file"]
data = bytearray(model_file.read_bytes())
data[len(data) // 2] ^= 1  # one bit of one byte
model_file.write_bytes(bytes(data))
try:
    load_bundle(str(tampered))
except ValueError as refusal:  # caught only to display the message
    print(refusal)
else:
    raise RuntimeError("the tampered bundle loaded")

## 6. The run record
A run is one Training call and its Evaluation, in `<roots.data>/<use case>/06_models/r007/`.
`create_run(env, resolved, entity, stage2)` (`runs.py`) takes the next number, creates the folder
exclusively and writes `run.json` into it, once: **one dict** of what the run is built from (the
run number, the use case and entity, who and when, the resolved config's identity, Training's
settings and their digest, the Stage 2 run tokens, the sources' digest, the framework version).
Nothing the run produces is in it: that goes to `training.json` and the manifest, later.

- **The number** is one above the highest among the existing run folders and the runs any audit
  log of the use case names, so a number is never given twice, also after its folder is deleted.
  Two people training at once get two numbers, since the folder is created exclusively.
- **Written first, never again:** `run.json` reserves the number before anything is trained and
  records what went in, so a later file can never change what the run was built from.
- **A folder without `training/` is incomplete:** a crashed run, or one still training.
  `list_runs` lists it with `trained` false, and its number stays taken while the folder exists.

`stage2` holds the run tokens of the EDA and Feature Checks runs the run is trained against;
Training passes the ones it reads, and here they come from the two stage jsons. The cell runs on a
copy of the environment whose data root is `TEMP`, so nothing is written into the use case's
`06_models/`; with no audit log under that root, the numbers start at `r001`. It creates two runs
to show the numbering.

`read_run` returns `run.json`; `list_runs` gives **one row per run folder, newest first** (the
run, who and when, the data definition, Training's and Evaluation's summaries once they exist,
and the states `trained`, `evaluated` and `signed_off`); `show_run` prints one run as plain text.

**Right looks like:** `r001` and `r002`, both incomplete; `run.json` naming this config's hashes
and the two Stage 2 tokens; each run folder holding `run.json` alone.

In [ ]:
temp_roots = env.roots.model_copy(update={"data": str(TEMP / "data")})
runs_env = env.model_copy(update={"roots": temp_roots})
stage2 = {
    stage: stage_json(Path(folder), stage)["run_token"] for stage in ("eda", "feature_checks")
}
first_run = create_run(runs_env, resolved, ENTITY, stage2)
second_run = create_run(runs_env, resolved, ENTITY, stage2)
record = read_run(runs_env, resolved, first_run)
display(pd.DataFrame({"value": {key: str(value)[:100] for key, value in record.items()}}))
display(sorted(str(path.relative_to(TEMP)) for path in (TEMP / "data").rglob("*")))
display(list_runs(runs_env, resolved, ENTITY))
print(show_run(runs_env, resolved, ENTITY, first_run))

## 7. A trained run: `training.json`
Sections 1 to 6 call the foundations on a small fixed fit. Sections 7 to 9 read what
`run_training` (`training/run.py`) wrote for a real run, and write nothing: run `04_training`
first. `find_run` (`runs.py`) picks the run Evaluation will read: the latest trained run of this
data definition with the current Training settings and sources (it says why when there is
none). `read_stage` (`artifacts.py`) reads `training/training.json` and every table it lists,
checking the format version first.

`training.json` is **one dict**, built by `_stage_meta` (`training/run.py`); `write_stage` adds
the stamp (`schema_version`, `stage`, `framework_version`, `written_at`, `tables`). The keys, by
group:
- **Identity:** `stage_identity`'s (`artifacts.py`: the hashes, `replayed_to`, Training's
  `settings` with their `settings_digest`, the `sources` with their `sources_digest`), `run_id`,
  and `stage2`, the EDA and Feature Checks run tokens the run was trained against.
- **Summary:** `summary`, the flat block `list_runs` and `show_run` display.
- **The search:** `best` (the family and its parameters, without the thread count), `search`
  (one row per family: `search_summary`, `training/views.py`) and `search_space` (each family's
  space as the search received it).
- **The rest:** `metrics` (`train_metrics`, `training/tables.py`), `final_features`, `sampling`
  (the sampling record), `folds` (one row per dealing and fold), `reproducible_to` (`model`, or
  `recipe` under a time budget), `warnings` and `timings`.

**Look at:** the keys with their types, then `summary` and `best`.

In [ ]:
run_id = find_run(env, resolved, ENTITY)
run_folder = models_dir(env, resolved, run_id)
meta, tables = read_stage(models_dir(env, resolved, run_id, "training"), "training")
display(
    pd.DataFrame(
        {
            "type": {key: type(value).__name__ for key, value in meta.items()},
            "value": {key: str(value)[:100] for key, value in meta.items()},
        }
    )
)
display(meta["summary"], meta["best"])

## 8. Training's tables
`read_stage` returns them by name; `training_views` (`training/views.py`) turns each into the
view `04_training` shows. Grain, columns and writer:
- **`trials`**, one row per family and trial: `trial` from 1, `fold_loss` (the fold metric as a
  loss, lower is better), `seconds`, `improving`, `configuration`. `read_trials`
  (`training/search.py`), from the search's log of each family.
- **`selection`**, one row per feature and rule that measured it: `value`, `threshold`,
  `outcome`, `represented_by`. `select_features` (`training/selection.py`).
- **`oof`**, one row per train row as trained: the keys, `cv_fold`, `target` and the
  out-of-fold `margin`, `score_raw` and `score`. `oof_table` (`training/tables.py`).
- **`explanation_sample`**, one row per sampled train row: the keys, the final features' raw
  values, `<feature>__contribution` per feature and `base_value`. `explanation_sample`
  (`training/tables.py`).
- **`calibration`**, one row per method considered, the bundle's `chosen`. `calibrate`
  (`training/calibrate.py`).
- **`signoff`**, one row per feature, by `section`. `signoff_table` (`training/ledger_run.py`).

**Right looks like:** each family ran its budget (`split_budget`, `training/search.py`; fewer
only under `training.time_budget_s`); as many out-of-fold rows as the first dealing's rows in
`meta["folds"]`, every margin finite; in the explanation sample, a row's contributions plus
`base_value` equal to its out-of-fold margin (both come from the fold model that did not see the
row's client); one calibration `chosen`; the sign-off's `model` rows are `final_features`.

In [ ]:
display(
    pd.DataFrame(
        {
            name: {"rows": len(table), "columns": ", ".join(table.columns)}
            for name, table in tables.items()
        }
    ).T
)
trials, oof, sample = tables["trials"], tables["oof"], tables["explanation_sample"]
final = meta["final_features"]
joined = sample.merge(oof[["unit_id", "period", "margin"]], on=["unit_id", "period"])
added = sample_contributions(joined, final).sum(axis=1) + joined["base_value"]
first_dealing = pd.DataFrame(meta["folds"]).query("dealing == 1")
model_rows = tables["signoff"].query("section == 'model'")["feature"]
checks = {
    "trials per family": trials.groupby("family", sort=False).size().to_dict(),
    "budget per family": split_budget(resolved),
    "oof rows equal the first dealing's": len(oof) == first_dealing["rows"].sum(),
    "every oof margin finite": bool(np.isfinite(oof["margin"]).all()),
    "contributions + base_value = oof margin": np.allclose(
        added, joined["margin"], rtol=1e-5, atol=1e-5
    ),
    "calibration methods chosen": int(tables["calibration"]["chosen"].sum()),
    "sign-off model rows = final_features": set(model_rows) == set(final),
}
display(pd.Series(checks).to_frame("check"))
display(trials.head(6), tables["selection"].head(10), oof.head(6), tables["calibration"])

## 9. The per-run ledger and `stage2/`
`read_run_ledger(run_folder)` (`training/ledger_run.py`) reads `ledger_run.parquet`, refusing
another `ledger_run_schema_version` (its sidecar `_ledger_run.json` holds it, with the run and
the Stage 2 run tokens). **One row per feature that entered selection**, in the encoder's
order:
- `selected` and `dropped_by`; per rule its `value`, `threshold` and `outcome` (`cluster_*`,
  `floor_*`, `prune_*`); `represents` and `protected` (a person's `keep`);
- the final model's `in_model`, `splits` and `gain_share`; the out-of-fold `contribution_share`
  and `contribution_rank`; `direction_model` and `direction_agrees` (against EDA's);
- the encoder's `levels_kept` and the train shares (`null_share`, `infinite_share`,
  `other_share`); `monotone` and `monotone_agrees`;
- `train_concordance` and `train_sign`, which Evaluation's single-feature baseline reads.

The model's columns are null for a feature outside the final list. Built by `build_run_ledger`,
saved by `save_run_ledger`; the sign-off table is built from it. `stage2/` is the run's own copy
of what it read from Stage 2 (`save_stage2`, `training/gate.py`): every candidate's ledger row
and, for the final features, EDA's `cells`, `edges`, `bins` and `levels`, so a later EDA run
never changes this run.

**Right looks like:** the selected rows are `final_features`, in their order; every dropped row
names its rule. **Look at:** the top features by contribution; the dropped ones.

In [ ]:
run_ledger = read_run_ledger(run_folder)
display(run_ledger.dtypes.to_frame("dtype").T)
display(run_ledger.sort_values("contribution_rank").head(10))
display(run_ledger.loc[~run_ledger["selected"]].head(10))
display(sorted(path.name for path in Path(run_folder, "stage2").iterdir()))
right = {
    "selected rows = final_features": run_ledger.loc[run_ledger["selected"], "feature"].tolist()
    == final,
    "every dropped row names its rule": bool(
        run_ledger.loc[~run_ledger["selected"], "dropped_by"].notna().all()
    ),
}
display(pd.Series(right).to_frame("check"))

## 10. An evaluated run: `evaluation.json`, its tables and the scores
Read-only, like sections 7 to 9: run `05_evaluation` on the run of section 7 first.
`read_stage` reads `evaluation/evaluation.json` and every table it lists.

`evaluation.json` is **one dict**, built by `_build_result` (`evaluation/run.py`): the identity
(`stage_identity`'s, `run_id`, `model_hash`, `validation_period`), `summary` (what `list_runs`
and `show_run` display), `base` (the unsampled train level), `eligible_rows`, `choices` (the
headline metrics, the single feature, the explanation cut, the effect features), Training's
`metrics`, `earlier_validations`, `drift_reference_schema_version`, `warnings` and `timings`.

The tables, by grain and writer:
- **`metrics`**, one row per metric, part, period (or pooled), stratum and level: `value`, `low`,
  `high`, `rows`, `events`. `metric_tables` (`evaluation/tables.py`), from the score cells.
- **`slices`**, **`curves`**, **`reliability`**, **`score_distribution`**, **`strata`**,
  **`pooling`**: what the core views draw, from the same cells (`metric_tables`).
- **`baselines`**, one row per scorer, part and metric. `compare_baselines`
  (`evaluation/baselines.py`).
- **`contributions`**, one row per explanation-sample row: the keys, the final features' raw
  values, `<feature>__contribution` per feature, `base_value`, then the scores and `waterfall`.
  `compute_contributions` (`evaluation/explain.py`); `05_evaluation` step 8 builds a
  `shap.Explanation` from it.
- **`features`**, **`permutation_importance`**, **`partial_dependence`**: `feature_table`,
  `permutation_importance`, `partial_dependence` (`evaluation/explain.py`); **`effects`**:
  `effect_table` (`evaluation/effects.py`).
- **`cells`**: the model's score cells of the actual rows (`_stored_cells`, `evaluation/run.py`).
- **`drift_*`**: the drift reference Monitoring reads (`build_drift_reference`,
  `evaluation/drift_reference.py`).
- **`training_signoff`**, **`training_trials`**, **`training_explanation_sample`**: Training's
  tables, carried so that `evaluation/` renders alone.

The **scores** (`scores/`, written by Spark: `score_parts` and `write_scores`,
`evaluation/scores.py`): one row per client, period and part, read here with Spark.

**Right looks like:** the run of section 7; each contribution row adding up to its margin (the
same check as section 8, on test rows); one `highest`, one `cut` and one `typical` waterfall
row; one scores row per client, period and part.

In [ ]:
evaluation_folder = models_dir(env, resolved, run_id, "evaluation")
eval_meta, eval_tables = read_stage(evaluation_folder, "evaluation")
display(
    pd.DataFrame(
        {
            "type": {key: type(value).__name__ for key, value in eval_meta.items()},
            "value": {key: str(value)[:100] for key, value in eval_meta.items()},
        }
    )
)
display(
    pd.DataFrame(
        {
            name: {"rows": len(table), "columns": ", ".join(table.columns)[:200]}
            for name, table in eval_tables.items()
        }
    ).T
)
metrics, contributions = eval_tables["metrics"], eval_tables["contributions"]
added = sample_contributions(contributions, final).sum(axis=1) + contributions["base_value"]
scores = spark.read.parquet(f"{evaluation_folder}/scores")
keyed = scores.select("unit_id", "period", "part")
checks = {
    "the run of section 7": eval_meta["run_id"] == run_id,
    "parts in metrics": sorted(metrics["part"].unique()),
    "contributions + base_value = margin": np.allclose(
        added, contributions["margin"], rtol=1e-4, atol=1e-4
    ),
    "waterfall rows": contributions["waterfall"].value_counts().to_dict(),
    "one scores row per client, period and part": keyed.distinct().count() == keyed.count(),
}
display(pd.Series(checks).to_frame("check"))
display(metrics.head(10), scores.groupBy("part").count().toPandas(), scores.limit(5).toPandas())

## What was shown
- **The training frame:** the train split of the window only, whole clients dealt to the folds,
  the converting clients spread evenly.
- **The encoder:** plain data fitted on train; three code rules, null kept, rare and unseen levels
  in `__other__`, nothing imputed.
- **The families:** one interface over four boosters; each saved as its library's own file, built
  back from the bytes with identical margins, its contributions adding up to the margin, the
  constraint held.
- **The bundle:** three files under one `model_hash`, checked at every load; one scoring path in
  pandas and in Spark; a changed byte refused.
- **The run record:** the number reserved and `run.json` written once, before anything is trained.
- **A trained run:** `training.json`, six tables and the per-run ledger, read from the run
  Evaluation will read; the explanation sample's contributions add up to the out-of-fold margin.
- **An evaluated run:** `evaluation.json`, its tables and the scores, read from the same run;
  the test contributions add up to the scored margin.

The last cell removes the temporary
folder and everything written in it.

In [ ]:
TEMP_DIR.cleanup()